## CARGA ENVIROMENT SPARK - CONEXIONES A SERVIDOR LOCAL + MINIO

In [ ]:
%%capture
import os
from pathlib import Path
from dotenv import load_dotenv
os.environ["SPARK_LOCAL_IP"] = "127.0.0.1"
os.environ["PYSPARK_SUBMIT_ARGS"] = (
    "--driver-memory 2g "
    "--packages org.apache.hadoop:hadoop-aws:3.5.0 "
    "pyspark-shell"
)
load_dotenv(Path.cwd() / "minio.env", override=True)
for nombre in ("MINIO_ENDPOINT", "MINIO_ACCESS_KEY", "MINIO_SECRET_KEY"):
    if not os.getenv(nombre):
        raise ValueError(f"Falta {nombre} en minio.env")

from pyspark.sql import SparkSession
spark = (
    SparkSession.builder
    .master("local[2]")
    .appName("Leer egresos")
    .config("spark.driver.host", "127.0.0.1")
    .config("spark.driver.bindAddress", "127.0.0.1")
    .config("spark.sql.files.maxPartitionBytes", "32m")
    .config("spark.hadoop.fs.s3a.input.stream.type", "classic")
    .config("spark.hadoop.fs.s3a.endpoint", os.environ["MINIO_ENDPOINT"])
    .config("spark.hadoop.fs.s3a.endpoint.region", "us-east-1")
    .config("spark.hadoop.fs.s3a.path.style.access", "true")
    .config("spark.hadoop.fs.s3a.connection.ssl.enabled", "false")
    .config("spark.hadoop.fs.s3a.access.key", os.environ["MINIO_ACCESS_KEY"])
    .config("spark.hadoop.fs.s3a.secret.key", os.environ["MINIO_SECRET_KEY"])
    .config(
        "spark.hadoop.fs.s3a.aws.credentials.provider",
        "org.apache.hadoop.fs.s3a.SimpleAWSCredentialsProvider"
    )
    .getOrCreate()
)
spark.conf.set("spark.sql.legacy.javaCharsets", "true")
spark.sparkContext.setLogLevel("ERROR")
spark._jvm.java.lang.Thread.currentThread().getContextClassLoader().loadClass(
    "org.apache.hadoop.fs.s3a.S3AFileSystem"
)

## INFERIR LOS CODIGOS Y SUS GLOSAS ASOCIADAS ... AGRUPO PARA OBTENER LAS LLAVE GLOSA O CODIGO

In [ ]:
from pyspark.sql import functions as F
df = (
    spark.read
    .option("header", "true")
    .option("sep", ";")
    .option("inferSchema", "false")
    .csv("s3a://doctorado/data/egresos/egresos_hospitalarios_2024.csv")
)
enfermedades = (
    df.select(
        F.regexp_extract(
            F.upper(F.trim(F.col("cau_cie10"))),
            r"^(\S+)", 1
        ).alias("codigo"),
        F.trim(
            F.regexp_replace(
                F.trim(F.col("cau_cie10")),
                r"^\S+\s*", ""
            )
        ).alias("nombre")
    )
    .filter(F.col("codigo").isin("G300", "G301", "G308", "G309"))
    .dropDuplicates(["codigo", "nombre"])
    .orderBy("codigo", "nombre")
)

df_enfermedades = enfermedades.select("codigo", "nombre")
pdf = enfermedades.toPandas()
display(pdf)


## AQUI GENERO FUNCION PARA LEER DESDE EL STORAGE + SPARK EL CSV POR ANIO DEPENDIENDO SI ES CODIGO O GLOSA

In [ ]:
from pyspark.sql import functions as F

def leer_egresos(
    anio,
    filtrar_por="codigo",
    codigos=None,
    catalogo=None,
    ruta_base="s3a://doctorado/data/egresos",
    encoding= "iso-8859-1"
):
    """Devuelve un DataFrame Spark filtrado por código o nombre."""
    if filtrar_por not in ("codigo", "nombre"):
        raise ValueError("filtrar_por debe ser 'codigo' o 'nombre'")

    if filtrar_por == "nombre" and catalogo is None:
        raise ValueError("Debes proporcionar catalogo para filtrar por nombre")

    if codigos is None:
        codigos = ["G300", "G301", "G308", "G309"]

    def normalizar(columna):
        return F.lower(
            F.trim(F.regexp_replace(columna, r"\s+", " "))
        )

    tabla = (
        spark.read
        .option("header", "true")
        .option("sep", ";")
        .option("inferSchema", "false")
        .option("encoding", encoding)
        .csv(
            f"{ruta_base.rstrip('/')}/"
            f"egresos_hospitalarios_{anio}.csv"
        )
        .withColumn(
            "codigo",
            F.regexp_replace(
                F.regexp_extract(
                    F.upper(F.trim(F.col("cau_cie10"))),
                    r"^([A-Z][0-9]{2}\.?[0-9A-Z]*)(?=\s|$)",
                    1
                ),
                r"\.",
                ""
            )
        )
        .withColumn(
            "nombre",
            F.trim(
                F.regexp_replace(
                    F.trim(F.col("cau_cie10")),
                    r"(?i)^[A-Z][0-9]{2}\.?[0-9A-Z]*\s+",
                    ""
                )
            )
        )
    )

    if filtrar_por == "codigo":
        return tabla.filter(F.col("codigo").isin(codigos))

    nombres = (
        catalogo
        .select(normalizar(F.col("nombre")).alias("_nombre_filtro"))
        .distinct()
    )

    return (
        tabla
        .withColumn("_nombre_filtro", normalizar(F.col("nombre")))
        .join(F.broadcast(nombres), on="_nombre_filtro", how="left_semi")
        .drop("_nombre_filtro")
    )

## CONSOLIDADO TABLA

### prov_ubi
* 01 Azuay
* 02 Bolívar
* 03 Cañar
* 04 Carchi
* 05 Cotopaxi
* 06 Chimborazo
* 07 El Oro
* 08 Esmeraldas
* 09 Guayas
* 10 Imbabura
* 11 Loja
* 12 Los Ríos
* 13 Manabí
* 14 Morona Santiago
* 15 Napo
* 16 Pastaza
* 17 Pichincha
* 18 Tungurahua
* 19 Zamora Chinchipe
* 20 Galápagos
* 21 Sucumbíos
* 22 Orellana
* 23 Santo Domingo de los Tsáchilas
* 24 Santa Elena

### area_ubi
* 1 Urbano
* 2 Rural

### sector
* 1 Público
* 2 Privado con fines de lucro
* 3 Privado sin fines de lucro

### sexo
* 1 Hombre
* 2 Mujer

### etnia
* 1 Indígena
* 2 Afroecuatoriano/a Afrodescendiente
* 3 Negro/a
* 4 Mulato/a
* 5 Montubio/a
* 6 Mestizo/a
* 7 Blanco/a
* 8 Otro/a
* 9 Ignorado/a

### con_egrpa
* 1 Vivo
* 2 Fallecido menos de 48 horas
* 3 Fallecido en 48 horas y más

In [ ]:
from functools import reduce
from pyspark.sql import functions as F

diccionarios = {
    "prov_ubi": {
        1: "Azuay", 2: "Bolívar", 3: "Cañar", 4: "Carchi",
        5: "Cotopaxi", 6: "Chimborazo", 7: "El Oro",
        8: "Esmeraldas", 9: "Guayas", 10: "Imbabura",
        11: "Loja", 12: "Los Ríos", 13: "Manabí",
        14: "Morona Santiago", 15: "Napo", 16: "Pastaza",
        17: "Pichincha", 18: "Tungurahua",
        19: "Zamora Chinchipe", 20: "Galápagos",
        21: "Sucumbíos", 22: "Orellana",
        23: "Santo Domingo de los Tsáchilas", 24: "Santa Elena"
    },
    "area_ubi": {
        1: "Urbano", 2: "Rural"
    },
    "sector": {
        1: "Público",
        2: "Privado con fines de lucro",
        3: "Privado sin fines de lucro"
    },
    "sexo": {
        1: "Hombre", 2: "Mujer"
    },
    "etnia": {
        1: "Indígena",
        2: "Afroecuatoriano/a Afrodescendiente",
        3: "Negro/a",
        4: "Mulato/a",
        5: "Montubio/a",
        6: "Mestizo/a",
        7: "Blanco/a",
        8: "Otro/a",
        9: "Ignorado/a"
    },
    "con_egrpa": {
        1: "Vivo",
        2: "Fallecido menos de 48 horas",
        3: "Fallecido en 48 horas y más"
    }
}

enfermedades_por_codigo = {
    "G300": "Enfermedad de Alzheimer de comienzo temprano",
    "G301": "Enfermedad de Alzheimer de comienzo tardío",
    "G308": "Otros tipos de enfermedad de Alzheimer",
    "G309": "Enfermedad de Alzheimer, no especificada"
}

configuracion_por_anio = {
    2015: {"filtro": "nombre", "encoding": "windows-1252", "diccionario": False},
    2016: {"filtro": "codigo", "encoding": "windows-1252", "diccionario": True},
    2017: {"filtro": "nombre", "encoding": "windows-1252", "diccionario": False},
    2018: {"filtro": "codigo", "encoding": "UTF-8", "diccionario": True},
    2019: {"filtro": "codigo", "encoding": "UTF-8", "diccionario": True},
    2020: {"filtro": "codigo", "encoding": "UTF-8", "diccionario": True},
    2021: {"filtro": "nombre", "encoding": "UTF-8", "diccionario": False},
    2022: {"filtro": "codigo", "encoding": "UTF-8", "diccionario": False},
    2023: {"filtro": "codigo", "encoding": "UTF-8", "diccionario": False},
    2024: {"filtro": "codigo", "encoding": "UTF-8", "diccionario": False},
}

correcciones_por_anio = {
    # Activa solo si sigue dañado después de leer con UTF-8:
    # 2024: {"sector": {"PÃºblico": "Público"}}
}

columnas = [
    "prov_ubi",
    "area_ubi",
    "sector",
    "sexo",
    "edad",
    "etnia",
    "anio_ingr",
    "dia_estad",
    "con_egrpa",
    "cau_cie10",
    "anio_ingesta",
]

spark.conf.set("spark.sql.legacy.javaCharsets", "true")


def crear_mapa(diccionario):
    return F.create_map(*[
        expresion
        for clave, valor in diccionario.items()
        for expresion in (F.lit(clave), F.lit(valor))
    ])


def normalizar_texto(columna):
    return F.lower(
        F.trim(F.regexp_replace(columna, r"\s+", " "))
    )


def extraer_codigo(columna):
    return F.regexp_replace(
        F.regexp_extract(
            F.upper(F.trim(columna)),
            r"^([A-Z][0-9]{2}\.?[0-9A-Z]*)(?=\s|$)",
            1
        ),
        r"\.",
        ""
    )


def extraer_nombre(columna):
    return F.trim(
        F.regexp_replace(
            F.trim(columna),
            r"(?i)^[A-Z][0-9]{2}\.?[0-9A-Z]*(?:\s+|$)",
            ""
        )
    )


# Catálogo independiente de las lecturas de otros años.
df_enfermedades = spark.createDataFrame(
    list(enfermedades_por_codigo.items()),
    ["codigo", "nombre"]
)


def leer_egresos(
    anio,
    filtrar_por="codigo",
    codigos=None,
    catalogo=None,
    ruta_base="s3a://doctorado/data/egresos",
    encoding="UTF-8"
):
    if filtrar_por not in ("codigo", "nombre"):
        raise ValueError("filtrar_por debe ser 'codigo' o 'nombre'")

    if codigos is None:
        codigos = list(enfermedades_por_codigo)

    tabla = (
        spark.read
        .option("header", "true")
        .option("sep", ";")
        .option("inferSchema", "false")
        .option("encoding", encoding)
        .csv(
            f"{ruta_base.rstrip('/')}/"
            f"egresos_hospitalarios_{anio}.csv"
        )
        .withColumn("codigo", extraer_codigo(F.col("cau_cie10")))
        .withColumn("nombre", extraer_nombre(F.col("cau_cie10")))
    )

    if filtrar_por == "codigo":
        return tabla.filter(F.col("codigo").isin(codigos))

    if catalogo is None:
        raise ValueError("Debes proporcionar un catálogo para filtrar por nombre")

    nombres_filtro = (
        catalogo
        .select(normalizar_texto(F.col("nombre")).alias("_nombre_filtro"))
        .distinct()
    )

    return (
        tabla
        .withColumn("_nombre_filtro", normalizar_texto(F.col("nombre")))
        .join(
            F.broadcast(nombres_filtro),
            on="_nombre_filtro",
            how="left_semi"
        )
        .drop("_nombre_filtro")
    )


def convertir_categorias(tabla):
    for columna, categorias in diccionarios.items():
        if columna not in tabla.columns:
            continue

        valor = F.trim(F.col(columna).cast("string"))

        codigo = F.when(
            valor.rlike(r"^\d{1,2}(?:\.0+)?$"),
            valor.cast("double").cast("int")
        )

        tabla = tabla.withColumn(
            columna,
            F.coalesce(
                F.element_at(crear_mapa(categorias), codigo),
                F.col(columna).cast("string")
            )
        )

    return tabla


def normalizar_enfermedad(tabla):
    valor = F.col("cau_cie10")
    codigo = extraer_codigo(valor)
    nombre = extraer_nombre(valor)

    nombres_canonicos = {
        " ".join(nombre.split()).lower(): nombre
        for nombre in enfermedades_por_codigo.values()
    }

    return tabla.withColumn(
        "cau_cie10",
        F.coalesce(
            F.element_at(crear_mapa(enfermedades_por_codigo), codigo),
            F.element_at(
                crear_mapa(nombres_canonicos),
                normalizar_texto(nombre)
            ),
            F.trim(valor)
        )
    )


def cargar_anios(configuracion, correcciones=None):
    if not configuracion:
        raise ValueError("Debes configurar al menos un año")

    correcciones = correcciones or {}
    dataframes = {}

    for anio, opciones in configuracion.items():
        tabla = leer_egresos(
            anio,
            filtrar_por=opciones["filtro"],
            catalogo=df_enfermedades,
            encoding=opciones["encoding"]
        )

        if opciones["diccionario"]:
            tabla = convertir_categorias(tabla)

        for columna, reemplazos in correcciones.get(anio, {}).items():
            tabla = tabla.replace(reemplazos, subset=[columna])

        tabla = normalizar_enfermedad(tabla)

        if "area_ubi" in tabla.columns:
            tabla = tabla.withColumn(
                "area_ubi",
                F.when(
                    normalizar_texto(F.col("area_ubi")) == "urbana",
                    F.lit("Urbano")
                ).otherwise(F.col("area_ubi"))
            )
        tabla = tabla.withColumn("anio_ingesta", F.lit(anio))
        dataframes[anio] = tabla
    unificado = reduce(
        lambda izquierda, derecha: izquierda.unionByName(
            derecha,
            allowMissingColumns=True
        ),
        dataframes.values()
    )

    return dataframes, unificado


dataframes, df_unificado = cargar_anios(
    configuracion_por_anio,
    correcciones_por_anio
)

df_final = df_unificado.select(*columnas)

df = df_final.limit(3000).toPandas()
df.head(10)

In [ ]:
import numpy as np
df= df.iloc[np.where(np.isin(df.anio_ingr.astype('int').tolist(),list(range(2015,2025))))]

In [ ]:
df

In [ ]:
df.groupby('etnia').count()

In [ ]:
import pandas as pd

df_en = df.copy()

# Unificar sector antes de traducir
df_en["sector"] = df_en["sector"].replace({
    "Privado con fines de lucro": "Privado",
    "Privado sin fines de lucro": "Privado"
})

# Traducir categorías; conservar valores que no estén en el mapa
traducciones = {
    "con_egrpa": {
        "Vivo": "Alive",
        "Fallecido": "Dead",
        "Muerto": "Dead"
    },
    "sexo": {
        "Hombre": "Male",
        "Mujer": "Female"
    },
    "etnia": {
            "Afroecuatoriano/a Afrodescendiente": "Afro-Ecuadorian/ Afro-descendant",
            "Negro/a": "Afro-Ecuadorian/ Afro-descendant",
            "Blanco/a": "White",
            "Ignorado/a": "Unknown",
            "Indígena": "Indigenous",
            "Mestizo/a": "Mestizo",
            "Mulato/a": "Mulatto",
            "Otro/a": "Other",
            "Sin información": "No information"
    },
    "area_ubi": {
        "Rural": "Rural",
        "Urbano": "Urban"
    },
    "sector": {
        "Privado": "Private",
        "Público": "Public"
    },
    "cau_cie10": {
        "Enfermedad de Alzheimer de comienzo temprano":
            "Alzheimer's disease with early onset",
        "Enfermedad de Alzheimer de comienzo tardío":
            "Alzheimer's disease with late onset",
        "Otros tipos de enfermedad de Alzheimer":
            "Other Alzheimer's disease",
        "Enfermedad de Alzheimer, no especificada":
            "Alzheimer's disease, unspecified"
    }
}

for columna, mapa in traducciones.items():
    df_en[columna] = df_en[columna].replace(mapa)

# Crear grupos de edad
edad = pd.to_numeric(df_en["edad"], errors="coerce")

df_en["Age groups"] = pd.cut(
    edad,
    bins=[0, 18, 45, 65, 80, float("inf")],
    labels=[
        "0-17 years", "18-44 years", "45-64 years",
        "65-79 years", "≥80 years"
    ],
    right=False
)

# Crear grupos de estancia
dias = pd.to_numeric(df_en["dia_estad"], errors="coerce")

df_en["Hospitalization groups"] = pd.cut(
    dias,
    bins=[1, 4, 15, 31, float("inf")],
    labels=["1-3 days", "4-14 days", "15-30 days", "≥31 days"],
    right=False
)

# Renombrar columnas
df_en = df_en.rename(columns={
    "prov_ubi": "Province",
    "area_ubi": "Area of residence",
    "sector": "Sector of health center",
    "sexo": "Sex",
    "edad": "Age",
    "etnia": "Ethnicity",
    "anio_ingr": "Year of admission",
    "dia_estad": "Days of Hospitalization",
    "con_egrpa": "Mortality",
    "cau_cie10": "Alzheimer's Diagnostic Categories (ICD-10)",
    "anio_ingesta": "Ingestion year"
})

In [ ]:
df_en

In [ ]:
import pandas as pd
from IPython.display import display

variables = {
    "Mortality": "Mortality",
    "Age groups": "Age groups",
    "Sex": "Sex",
    "Ethnicity": "Ethnicity",
    "Area of residence": "Area of residence",
    "Sector of health center": "Sector of health center",
    "Alzheimer's Diagnostic Categories (ICD-10)":
        "Alzheimer's Diagnostic Categories (ICD-10)",
    "Days of Hospitalization": "Hospitalization groups",
    "Year of admission": "Year of admission"
}

filas = []
encabezados = []

for titulo, columna in variables.items():
    if filas:
        filas.append(["", "", ""])  # Separación entre variables

    encabezados.append(len(filas))
    filas.append([titulo, "", ""])

    serie = df_en[columna]
    frecuencias = serie.value_counts(dropna=False, sort=False)

    if columna == "Year of admission":
        frecuencias = frecuencias.sort_index()

    for categoria, n in frecuencias.items():
        etiqueta = "Missing" if pd.isna(categoria) else str(categoria)
        filas.append([etiqueta, int(n), f"{n / len(df_en) * 100:.1f}"])

tabla = pd.DataFrame(filas, columns=["Variables", "n", "%"])

def formato_fila(fila):
    if fila.name in encabezados:
        return ["font-weight: bold;"] * len(fila)
    return ["padding-left: 16px;", "", ""]
estilo = (
    tabla.style
    .hide(axis="index")
    .apply(formato_fila, axis=1)
    .set_table_styles([
        {
            "selector": "",
            "props": [
                ("background-color", "white"),
                ("color", "black"),
                ("border-collapse", "collapse"),
                ("font-family", "'Times New Roman', serif"),
                ("font-size", "14px"),
                ("min-width", "600px")
            ]
        },
        {
            "selector": "th, td",
            "props": [
                ("background-color", "white"),
                ("color", "black")
            ]
        },
        {
            "selector": "thead th",
            "props": [
                ("border-top", "1px solid black"),
                ("border-bottom", "1px solid black"),
                ("font-weight", "normal"),
                ("text-align", "right")
            ]
        },
        {
            "selector": "thead th:first-child",
            "props": [("text-align", "left")]
        },
        {
            "selector": "tbody td",
            "props": [
                ("border", "none"),
                ("text-align", "right"),
                ("padding", "2px 8px")
            ]
        },
        {
            "selector": "tbody td:first-child",
            "props": [("text-align", "left")]
        },
        {
            "selector": "tbody tr:hover td",
            "props": [
                ("background-color", "white"),
                ("color", "black")
            ]
        },
        {
            "selector": "tbody tr:last-child td",
            "props": [("border-bottom", "1px solid black")]
        }
    ])
)

display(estilo)

In [ ]:
%pip install dataframe_image


In [ ]:
import dataframe_image as dfi
from IPython.display import display

# Aplicar formato original
estilo = (
    tabla.style
    .hide(axis="index")
    .apply(formato_fila, axis=1)
    .set_table_styles([
        {
            "selector": "",
            "props": [
                ("background-color", "white"),
                ("color", "black"),
                ("border-collapse", "collapse"),
                ("font-family", "'Times New Roman', serif"),
                ("font-size", "14px"),
                ("min-width", "600px")
            ]
        },
        {
            "selector": "th, td",
            "props": [
                ("background-color", "white"),
                ("color", "black")
            ]
        },
        {
            "selector": "thead th",
            "props": [
                ("border-top", "1px solid black"),
                ("border-bottom", "1px solid black"),
                ("font-weight", "normal"),
                ("text-align", "right")
            ]
        },
        {
            "selector": "thead th:first-child",
            "props": [
                ("text-align", "left")
            ]
        },
        {
            "selector": "tbody td",
            "props": [
                ("border", "none"),
                ("text-align", "right"),
                ("padding", "2px 8px")
            ]
        },
        {
            "selector": "tbody td:first-child",
            "props": [
                ("text-align", "left")
            ]
        },
        {
            "selector": "tbody tr:hover td",
            "props": [
                ("background-color", "white"),
                ("color", "black")
            ]
        },
        {
            "selector": "tbody tr:last-child td",
            "props": [
                ("border-bottom", "1px solid black")
            ]
        }
    ])
)

# Mostrar tabla
display(estilo)

# Exportar a PNG
dfi.export(
    estilo,
    "tabla_alzheimer.png",
    table_conversion="chrome",
    dpi=300
)

print("Tabla exportada correctamente.")

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style="whitegrid", context="paper", font_scale=1.2)

age = pd.DataFrame({
    "Age group": ["0–17", "18–44", "45–64", "65–79", "≥80"],
    "Patients": [0, 16, 69, 472, 1118]
})

sex = pd.DataFrame({
    "Sex": ["Male", "Female"],
    "Patients": [658, 1017]
})

fig, axes = plt.subplots(1, 2, figsize=(12, 5))

# Age distribution
sns.barplot(
    data=age,
    x="Age group",
    y="Patients",
    color="#4472C4",
    ax=axes[0]
)

axes[0].set_title("A. Age distribution", fontweight="bold")
axes[0].set_xlabel("Age group (years)")
axes[0].set_ylabel("Number of patients")

for container in axes[0].containers:
    axes[0].bar_label(container, padding=3)

# Sex distribution
colors = ["#4F81BD", "#C0507A"]

axes[1].pie(
    sex["Patients"],
    labels=sex["Sex"],
    autopct="%1.1f%%",
    startangle=90,
    colors=colors,
    wedgeprops={"edgecolor": "white"}
)

axes[1].set_title("B. Sex distribution", fontweight="bold")

plt.suptitle(
    "Demographic characteristics of hospitalized Alzheimer's patients",
    fontsize=14,
    fontweight="bold"
)

plt.tight_layout()
plt.show()

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

years = np.arange(2015, 2025)

hospitalizations = np.array([
    177, 151, 169, 185, 169,
    122, 162, 188, 169, 183
])

fig, ax = plt.subplots(figsize=(11, 5))

ax.plot(
    years,
    hospitalizations,
    marker="o",
    linewidth=2.5,
    markersize=7,
    color="#21618C",
    label="Hospitalizations"
)

ax.fill_between(
    years,
    hospitalizations,
    alpha=0.12,
    color="#21618C"
)

# Linear trend
coef = np.polyfit(years, hospitalizations, 1)
trend = np.polyval(coef, years)

ax.plot(
    years,
    trend,
    linestyle="--",
    color="#C0392B",
    linewidth=2,
    label="Linear trend"
)

for x, y in zip(years, hospitalizations):
    ax.annotate(
        str(y),
        (x, y),
        textcoords="offset points",
        xytext=(0, 9),
        ha="center"
    )

ax.set_title(
    "Annual hospitalizations for Alzheimer's disease (2015–2024)",
    fontsize=14,
    fontweight="bold"
)

ax.set_xlabel("Year")
ax.set_ylabel("Number of hospitalizations")
ax.set_xticks(years)

ax.legend(frameon=False)

sns.despine()
plt.tight_layout()
plt.show()


In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd

mortality = pd.DataFrame({
    "Category": ["Alive", "Dead"],
    "Percentage": [84.4, 15.6]
})

hospital_stay = pd.DataFrame({
    "Category": ["1–3 days", "4–14 days", "15–30 days", "≥31 days"],
    "Percentage": [29.3, 46.0, 10.8, 13.9]
})

diagnosis = pd.DataFrame({
    "Category": [
        "Unspecified",
        "Late onset",
        "Other Alzheimer's",
        "Early onset"
    ],
    "Percentage": [81.6, 9.9, 7.3, 1.3]
})

fig, axes = plt.subplots(1, 3, figsize=(16, 5))

datasets = [mortality, hospital_stay, diagnosis]

titles = [
    "A. Mortality",
    "B. Length of hospital stay",
    "C. Alzheimer's diagnosis"
]

colors = ["#C0504D", "#4F81BD", "#70AD47"]

for ax, df, title, color in zip(
    axes, datasets, titles, colors
):

    sns.barplot(
        data=df,
        y="Category",
        x="Percentage",
        color=color,
        ax=ax
    )

    ax.set_title(title, fontweight="bold")
    ax.set_xlabel("Percentage (%)")
    ax.set_ylabel("")
    ax.set_xlim(0, 100)

    for container in ax.containers:
        ax.bar_label(
            container,
            fmt="%.1f%%",
            padding=4
        )

plt.suptitle(
    "Clinical characteristics of Alzheimer's hospitalizations",
    fontsize=15,
    fontweight="bold"
)

plt.tight_layout()
plt.show()

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

ethnicity = pd.DataFrame({
    "Ethnicity": [
        "Mestizo",
        "Unknown",
        "Negro/a",
        "Other",
        "White",
        "Indigenous",
        "Mulatto",
        "Afro-Ecuadorian / Afro-descendant",
        "No information"
    ],
    "Patients": [
        1383, 111, 5, 86, 9, 14, 3, 3, 61
    ]
})

ethnicity["Percentage"] = (
    ethnicity["Patients"] / ethnicity["Patients"].sum() * 100
)

ethnicity = ethnicity.sort_values(
    "Percentage",
    ascending=True
)

sns.set_theme(style="whitegrid", context="paper", font_scale=1.2)

fig, ax = plt.subplots(figsize=(11, 6))

colors = [
    "#21618C" if x == "Mestizo" else "#A9CCE3"
    for x in ethnicity["Ethnicity"]
]

bars = ax.barh(
    ethnicity["Ethnicity"],
    ethnicity["Percentage"],
    color=colors,
    edgecolor="white"
)

for bar, value in zip(bars, ethnicity["Percentage"]):
    ax.text(
        bar.get_width() + 0.5,
        bar.get_y() + bar.get_height()/2,
        f"{value:.1f}%",
        va="center",
        fontsize=10
    )

ax.set_title(
    "Ethnic distribution of hospitalized Alzheimer's patients",
    fontsize=14,
    fontweight="bold"
)

ax.set_xlabel("Percentage of patients (%)")
ax.set_ylabel("Ethnicity")

ax.set_xlim(0, 95)

sns.despine(left=True, bottom=True)

plt.tight_layout()
plt.show()

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

categories = [
    "Area of residence",
    "Healthcare sector"
]

data = [
    [91.4, 8.6],
    [62.6, 37.4]
]

labels = [
    ["Urban", "Rural"],
    ["Public", "Private"]
]

colors = [
    ["#2874A6", "#85C1E9"],
    ["#239B56", "#A9DFBF"]
]

fig, ax = plt.subplots(figsize=(11, 5))

y_positions = np.arange(len(categories))

for i, values in enumerate(data):

    left = 0

    for j, value in enumerate(values):

        ax.barh(
            i,
            value,
            left=left,
            color=colors[i][j],
            height=0.45,
            edgecolor="white",
            linewidth=2
        )

        ax.text(
            left + value / 2,
            i,
            f"{labels[i][j]}\n{value:.1f}%",
            ha="center",
            va="center",
            fontsize=11,
            fontweight="bold",
            color="black"
        )

        left += value

ax.set_yticks(y_positions)
ax.set_yticklabels(categories)

ax.set_xlim(0, 100)
ax.set_xlabel("Percentage (%)")

ax.set_title(
    "Distribution by residence and healthcare sector",
    fontsize=14,
    fontweight="bold"
)

ax.invert_yaxis()

sns.despine(left=True)

plt.tight_layout()
plt.show()

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

diagnosis = pd.DataFrame({
    "Diagnosis": [
        "Unspecified Alzheimer's",
        "Late-onset Alzheimer's",
        "Other Alzheimer's",
        "Early-onset Alzheimer's"
    ],
    "Patients": [
        1366, 165, 122, 22
    ]
})

diagnosis = diagnosis.sort_values(
    "Patients",
    ascending=False
)

diagnosis["Percentage"] = (
    diagnosis["Patients"] /
    diagnosis["Patients"].sum() * 100
)

diagnosis["Cumulative"] = (
    diagnosis["Percentage"].cumsum()
)

fig, ax1 = plt.subplots(figsize=(11, 6))

bars = ax1.bar(
    diagnosis["Diagnosis"],
    diagnosis["Patients"],
    color="#4472C4",
    alpha=0.85
)

ax1.set_ylabel(
    "Number of hospitalizations",
    color="#21618C"
)

ax1.set_xlabel("Diagnostic category")

ax1.tick_params(
    axis="x",
    rotation=20
)

for bar in bars:
    height = bar.get_height()

    ax1.annotate(
        f"{int(height)}",
        xy=(
            bar.get_x() + bar.get_width()/2,
            height
        ),
        xytext=(0, 5),
        textcoords="offset points",
        ha="center"
    )

# Secondary axis
ax2 = ax1.twinx()

ax2.plot(
    diagnosis["Diagnosis"],
    diagnosis["Cumulative"],
    color="#C0392B",
    marker="o",
    linewidth=2.5
)

ax2.axhline(
    80,
    color="gray",
    linestyle="--",
    alpha=0.7,
    label="80% threshold"
)

for i, value in enumerate(diagnosis["Cumulative"]):
    ax2.annotate(
        f"{value:.1f}%",
        (i, value),
        textcoords="offset points",
        xytext=(0, 9),
        ha="center",
        color="#C0392B"
    )

ax2.set_ylabel("Cumulative percentage (%)")
ax2.set_ylim(0, 110)

ax2.legend(loc="center right", frameon=False)

plt.title(
    "Pareto analysis of Alzheimer's diagnostic categories",
    fontsize=14,
    fontweight="bold"
)

plt.tight_layout()
plt.show()